<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the libraries needed to run the notebook. Riskfolio-Lib handles portfolio optimization, and OpenBB provides free access to market data.

In [ ]:
!pip install riskfolio-lib openbb

OpenBB Terminal SDK requires additional setup beyond a simple pip install. Follow the official OpenBB installation guide for your operating system, as it depends on system-level dependencies and may need a separate environment.

## Imports and setup

openbb provides a unified interface for pulling stock screener results and historical price data from free sources. riskfolio (imported as rp) is a portfolio optimization library that implements risk parity and other allocation strategies out of the box.

In [ ]:
from openbb import obb
import riskfolio as rp

## Screen and retrieve stock data

This pulls a list of stocks currently making new highs, then filters for U.S.-listed stocks priced above $15 to focus on liquid, investable names.

In [ ]:
performance = obb.equity.screener().to_df()
port_data = performance[
    (performance.price > 25) & (performance.exchange == "NYQ")
]

Starting from a screener instead of a hand-picked list gives us a repeatable, data-driven universe of stocks. The price and country filters remove penny stocks and foreign listings that could introduce noise or liquidity problems into our risk calculations.

We extract the ticker symbols and download daily price history for each one over a seven-year window using OpenBB's data API.

In [ ]:
symbols = port_data.symbol.tolist()
data = obb.equity.price.historical(
    symbols,
    start_date="2016-01-01",
    end_date="2022-12-31",
).to_df()
data = data.pivot(columns="symbol", values="close").dropna()

This single call replaces the tedious process of downloading individual CSV files from multiple websites. Seven years of data gives the optimizer enough history to estimate volatility and correlations reliably, covering both calm and turbulent market periods.

## Optimize portfolio with risk parity

We convert prices to daily percentage returns and drop any stock that has missing data, since gaps would distort the covariance matrix the optimizer relies on.

In [ ]:
returns = data.pct_change()[1:]
returns.dropna(how="any", axis=1, inplace=True)

Dropping columns with any NaN values is aggressive but necessary here. A single missing day in one stock's history can silently break the covariance estimate and produce nonsensical portfolio weights. Cleaning the data before optimization is a habit that saves hours of debugging later.

We create a Portfolio object, estimate expected returns and the covariance matrix from historical data, then run the risk parity optimization with a minimum daily return constraint of 0.08%.

In [ ]:
port = rp.Portfolio(returns=returns)

In [ ]:
port.assets_stats(method_mu='hist', method_cov='hist')

In [ ]:
w_rp_c = port.rp_optimization(
    model="Classic",
    rm="MV",
    hist=True,
    rf=0,
    b=None,
)

The d=0.94 parameter applies exponential weighting so that recent returns matter more than older ones when estimating volatility. Setting lowerret adds the kind of minimum return requirement that professional managers use to prevent the optimizer from parking everything in the lowest-risk, lowest-return assets. The output w_rp_c is a DataFrame of portfolio weights where each stock contributes roughly the same amount of overall risk.

## Visualize and convert weights to shares

These two plots show the portfolio allocation as a pie chart and the risk contribution of each stock as a bar chart, letting us visually confirm that risk is spread evenly.

In [ ]:
ax = rp.plot_pie(w=w_rp_c)

In [ ]:
ax = rp.plot_risk_con(
    w_rp_c,
    cov=port.cov,
    returns=port.returns,
    rm="MV",
    rf=0,
)

The risk contribution chart is the one that matters most. If every bar is roughly the same height, the optimizer did its job and no single stock dominates our portfolio's volatility. In an equal-dollar portfolio, the most volatile stock would tower over the rest in this chart.

We translate the abstract percentage weights into a concrete trading plan by multiplying by a $10,000 portfolio value and dividing by each stock's last closing price to get whole share counts.

In [ ]:
port_val = 10_000
w_rp_c["invest_amt"] = w_rp_c * port_val
w_rp_c["last_price"] = data.iloc[-1]
w_rp_c["shares"] = (
    w_rp_c.invest_amt / w_rp_c.last_price
).astype(int)

This final step bridges the gap between theory and execution. Optimizer output is always in decimal weights, but you buy stocks in whole shares. Casting to integers with astype(int) rounds down, which means we'll have a small amount of cash left over rather than accidentally overspending our budget.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.